### Задание 1. Функция vs процедура в обработке ошибок

1. Процедура `log_error(path: str, reason: str) -> None` — логирует ошибку через `logging.warning`.
2. Функция `parse_int_safe(value: str) -> int | None` — возвращает число или `None` при `ValueError`.

In [3]:
import logging

logger = logging.getLogger("task1")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")


def log_error(path: str, reason: str) -> None:
    """Процедура: логирует ошибку и ничего не возвращает."""
    logger.warning("Ошибка в %s: %s", path, reason)


def parse_int_safe(value: str) -> int | None:
    """Функция: возвращает int или None при ValueError."""
    try:
        return int(value)
    except ValueError:
        return None


log_error("data.json", "не найден")
print(parse_int_safe("42"))    # 42
print(parse_int_safe("abc"))   # None

WARNING task1: Ошибка в data.json: не найден


42
None


### Задание 2. Все виды исключений

Для каждого выражения предскажите тип исключения, затем проверьте в try/except. Покажите `__mro__` и объясните, от кого наследуется.

```python
1 / 0
int("abc")
[1, 2][5]
{"a": 1}["b"]
"5" + 5
open("no_such_file.txt")
```

In [4]:
expressions = [
    ("1 / 0", lambda: 1 / 0),
    ("int('abc')", lambda: int("abc")),
    ("[1, 2][5]", lambda: [1, 2][5]),
    ("{'a': 1}['b']", lambda: {"a": 1}["b"]),
    ("'5' + 5", lambda: "5" + 5),
    ("open('no_such_file.txt')", lambda: open("no_such_file.txt")),
]

for label, func in expressions:
    try:
        func()
    except Exception as exc:
        print(f"{label:30s} -> {type(exc).__name__}")
        print("    MRO:", [c.__name__ for c in type(exc).__mro__])

1 / 0                          -> ZeroDivisionError
    MRO: ['ZeroDivisionError', 'ArithmeticError', 'Exception', 'BaseException', 'object']
int('abc')                     -> ValueError
    MRO: ['ValueError', 'Exception', 'BaseException', 'object']
[1, 2][5]                      -> IndexError
    MRO: ['IndexError', 'LookupError', 'Exception', 'BaseException', 'object']
{'a': 1}['b']                  -> KeyError
    MRO: ['KeyError', 'LookupError', 'Exception', 'BaseException', 'object']
'5' + 5                        -> TypeError
    MRO: ['TypeError', 'Exception', 'BaseException', 'object']
open('no_such_file.txt')       -> FileNotFoundError
    MRO: ['FileNotFoundError', 'OSError', 'Exception', 'BaseException', 'object']


### Задание 3. try / except / else / finally

Реализуйте `safe_divide(a, b) -> float | None`:
- в `try` — деление;
- в `except ZeroDivisionError` — лог и `None`;
- в `else` — сообщение об успехе;
- в `finally` — сообщение «операция завершена».

Продемонстрируйте: `(10, 2)`, `(10, 0)`, `(10, "x")`.

In [7]:
def safe_divide(a, b) -> float | None:
    try:
        result = a / b
    except ZeroDivisionError as exc:
        logger.warning("Деление на ноль: %s", exc)
        return None
    except TypeError as exc:
        logger.warning("Неверный тип: %s", exc)
        return None
    else:
        logger.info("Успешно: %s / %s = %s", a, b, result)
        return result
    finally:
        logger.info("Операция завершена")


print(safe_divide(10, 2))    # 5.0
print(safe_divide(10, 0))    # None
print(safe_divide(10, "x"))  # None

INFO task1: Успешно: 10 / 2 = 5.0
INFO task1: Операция завершена
WARNING task1: Деление на ноль: division by zero
INFO task1: Операция завершена
WARNING task1: Неверный тип: unsupported operand type(s) for /: 'int' and 'str'
INFO task1: Операция завершена


5.0
None
None


### Задание 4. Своя иерархия исключений

Создайте `IngestError`, `ConfigError`, `DocumentError`. Продемонстрируйте, что `except IngestError` ловит обоих наследников.

In [8]:
class IngestError(Exception):
    """Базовая ошибка конвейера импорта."""


class ConfigError(IngestError):
    """Настройки невалидны."""


class DocumentError(IngestError):
    """Проблема с конкретным документом."""

    def __init__(self, path, reason: str) -> None:
        super().__init__(f"{path}: {reason}")
        self.path = path
        self.reason = reason


for exc in (ConfigError("нет каталога"), DocumentError("a.json", "битый JSON")):
    try:
        raise exc
    except IngestError as e:            # ловит обоих наследников
        print(type(e).__name__, "->", e)

ConfigError -> нет каталога
DocumentError -> a.json: битый JSON


### Задание 5. LBYL vs EAFP

Реализуйте чтение файла двумя способами. Объясните, в каком случае EAFP надёжнее.

In [9]:
import os

path = "no_such_file.txt"

# LBYL
if os.path.exists(path):
    with open(path, encoding="utf-8") as f:
        data = f.read()
else:
    print("LBYL: файла нет")

# EAFP
try:
    with open(path, encoding="utf-8") as f:
        data = f.read()
except FileNotFoundError:
    print("EAFP: файла нет")

LBYL: файла нет
EAFP: файла нет
